In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    '/content/zomato.csv',
    encoding='latin1',
    engine='python',     # 🔥 KEY FIX
    on_bad_lines='skip'  # skip problematic rows
)

df.head()

,url,address,name,online_order,book_table,rate,votes,phone,location,rest_type,dish_liked,cuisines,approx_cost(for two people),reviews_list,menu_item,listed_in(type),listed_in(city)
0,https://www.zomato.com/bangalore/jalsa-banasha...,"942, 21st Main Road, 2nd Stage, Banashankari, ...",Jalsa,Yes,Yes,4.1/5,775,080 42297555\r\n+91 9743772233,Banashankari,Casual Dining,"Pasta, Lunch Buffet, Masala Papad, Paneer Laja...","North Indian, Mughlai, Chinese",800,"[('Rated 4.0', 'RATED\n A beautiful place to ...",[],Buffet,Banashankari
1,https://www.zomato.com/bangalore/spice-elephan...,"2nd Floor, 80 Feet Road, Near Big Bazaar, 6th ...",Spice Elephant,Yes,No,4.1/5,787,080 41714161,Banashankari,Casual Dining,"Momos, Lunch Buffet, Chocolate Nirvana, Thai G...","Chinese, North Indian, Thai",800,"[('Rated 4.0', 'RATED\n Had been here for din...",[],Buffet,Banashankari
2,https://www.zomato.com/SanchurroBangalore?cont...,"1112, Next to KIMS Medical College, 17th Cross...",San Churro Cafe,Yes,No,3.8/5,918,+91 9663487993,Banashankari,"Cafe, Casual Dining","Churros, Cannelloni, Minestrone Soup, Hot Choc...","Cafe, Mexican, Italian",800,"[('Rated 3.0', ""RATED\n Ambience is not that ...",[],Buffet,Banashankari
3,https://www.zomato.com/bangalore/addhuri-udupi...,"1st Floor, Annakuteera, 3rd Stage, Banashankar...",Addhuri Udupi Bhojana,No,No,3.7/5,88,+91 9620009302,Banashankari,Quick Bites,Masala Dosa,"South Indian, North Indian",300,"[('Rated 4.0', ""RATED\n Great food and proper...",[],Buffet,Banashankari
4,https://www.zomato.com/bangalore/grand-village...,"10, 3rd Floor, Lakshmi Associates, Gandhi Baza...",Grand Village,No,No,3.8/5,166,+91 8026612447\r\n+91 9901210005,Basavanagudi,Casual Dining,"Panipuri, Gol Gappe","North Indian, Rajasthani",600,"[('Rated 4.0', 'RATED\n Very good restaurant ...",[],Buffet,Banashankari


In [4]:
df = df[['name', 'location', 'cuisines', 'rate', 'votes', 'approx_cost(for two people)']]

df.head()

,name,location,cuisines,rate,votes,approx_cost(for two people)
0,Jalsa,Banashankari,"North Indian, Mughlai, Chinese",4.1/5,775,800
1,Spice Elephant,Banashankari,"Chinese, North Indian, Thai",4.1/5,787,800
2,San Churro Cafe,Banashankari,"Cafe, Mexican, Italian",3.8/5,918,800
3,Addhuri Udupi Bhojana,Banashankari,"South Indian, North Indian",3.7/5,88,300
4,Grand Village,Basavanagudi,"North Indian, Rajasthani",3.8/5,166,600


In [5]:
df.isnull().sum()

,0
name,0
location,4
cuisines,13
rate,2706
votes,0
approx_cost(for two people),56


In [6]:
df.dropna(inplace=True)

In [7]:
df['rate'] = df['rate'].str.replace('/5', '')
df = df[df['rate'] != 'NEW']
df = df[df['rate'] != '-']

df['rate'] = df['rate'].astype(float)

In [8]:
df['approx_cost(for two people)'] = df['approx_cost(for two people)'].astype(str)
df['approx_cost(for two people)'] = df['approx_cost(for two people)'].str.replace(',', '')
df['approx_cost(for two people)'] = df['approx_cost(for two people)'].astype(float)

In [9]:
df['cuisines'] = df['cuisines'].str.lower()

In [10]:
df.head()
df.shape

(14378, 6)

In [11]:
def recommend_restaurants(location, cuisine, max_cost):

    # Filter based on user input
    filtered = df[
        (df['location'].str.lower() == location.lower()) &
        (df['cuisines'].str.contains(cuisine.lower())) &
        (df['approx_cost(for two people)'] <= max_cost)
    ]

    # Sort by rating and votes (important!)
    filtered = filtered.sort_values(by=['rate', 'votes'], ascending=False)

    return filtered[['name', 'cuisines', 'rate', 'votes', 'approx_cost(for two people)']].head(10)

In [12]:
recommend_restaurants('Banashankari', 'north indian', 1000)

,name,cuisines,rate,votes,approx_cost(for two people)
480,Ayodhya Upachar,"south indian, north indian, chinese, street food",4.3,734,200.0
634,Ayodhya Upachar,"south indian, north indian, chinese, street food",4.3,734,200.0
34,Faasos,"north indian, biryani, fast food",4.2,415,500.0
694,Faasos,"north indian, biryani, fast food",4.2,415,500.0
2607,Faasos,"north indian, biryani, fast food",4.2,415,500.0
3457,Mini Punjabi Dhaba,north indian,4.2,325,350.0
191,Mini Punjabi Dhaba,north indian,4.2,287,350.0
556,Mini Punjabi Dhaba,north indian,4.2,287,350.0
2624,Mini Punjabi Dhaba,north indian,4.2,287,350.0
60,Peppy Peppers,"italian, north indian, mexican",4.2,244,800.0


In [13]:
pd.set_option('display.max_colwidth', None)

In [14]:
recommend_restaurants('Banashankari', 'chinese', 800)
recommend_restaurants('Basavanagudi', 'south indian', 500)

,name,cuisines,rate,votes,approx_cost(for two people)
3335,Brahmin's Coffee Bar,south indian,4.8,2679,100.0
3333,Mavalli Tiffin Room (MTR),south indian,4.5,2896,250.0
769,Vidyarthi Bhavan,south indian,4.4,4460,150.0
3334,Vidyarthi Bhavan,south indian,4.4,4460,150.0
2866,By 2 Coffee,south indian,4.3,316,100.0
284,South Kitchen,south indian,4.3,275,100.0
800,South Kitchen,south indian,4.3,275,100.0
3014,South Kitchen,south indian,4.3,275,100.0
3349,South Kitchen,south indian,4.3,275,100.0
294,Puliyogare Point,south indian,4.2,444,100.0


In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words='english')

tfidf_matrix = tfidf.fit_transform(df['cuisines'])

In [16]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [17]:
df = df.reset_index(drop=True)

In [18]:
def recommend_similar(name):

    if name not in df['name'].values:
        return "Restaurant not found"

    idx = df[df['name'] == name].index[0]

    similarity_scores = list(enumerate(cosine_sim[idx]))

    similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)

    similarity_scores = similarity_scores[1:11]  # top 10 similar

    restaurant_indices = [i[0] for i in similarity_scores]

    return df.iloc[restaurant_indices][['name', 'cuisines', 'rate']]

In [19]:
recommend_similar("Vidyarthi Bhavan")

,name,cuisines,rate
41,Havyaka Mess,south indian,3.9
66,Namma Brahmin's Idli,south indian,3.6
74,Sri Guru Kottureshwara Davangere Benne Dosa,south indian,4.1
94,Kidambi's Kitchen,south indian,3.5
95,Mane Thindi,south indian,3.7
111,Ruchi Maayaka,south indian,3.3
114,Vijayalakshmi,south indian,3.9
141,Bhattara Bhojana,south indian,3.6
153,By 2 Coffee,south indian,3.9
155,Davangere Butter Dosa Hotel,south indian,3.8


In [20]:
recommend_similar("Brahmin's Coffee Bar")

,name,cuisines,rate
41,Havyaka Mess,south indian,3.9
66,Namma Brahmin's Idli,south indian,3.6
74,Sri Guru Kottureshwara Davangere Benne Dosa,south indian,4.1
94,Kidambi's Kitchen,south indian,3.5
95,Mane Thindi,south indian,3.7
111,Ruchi Maayaka,south indian,3.3
114,Vijayalakshmi,south indian,3.9
141,Bhattara Bhojana,south indian,3.6
153,By 2 Coffee,south indian,3.9
155,Davangere Butter Dosa Hotel,south indian,3.8


In [21]:
df = df.drop_duplicates(subset='name')
df = df.reset_index(drop=True)

df.shape

(4693, 6)

In [22]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['cuisines'])

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [23]:
def recommend_smart(name, min_rating=3.5):

    if name not in df['name'].values:
        return "Restaurant not found"

    idx = df[df['name'] == name].index[0]

    similarity_scores = list(enumerate(cosine_sim[idx]))
    similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)

    similarity_scores = similarity_scores[1:20]  # take more candidates

    restaurant_indices = [i[0] for i in similarity_scores]

    result = df.iloc[restaurant_indices]

    # Apply rating filter
    result = result[result['rate'] >= min_rating]

    # Sort by rating + votes (real-world logic)
    result = result.sort_values(by=['rate', 'votes'], ascending=False)

    return result[['name', 'cuisines', 'rate', 'votes']].head(10)

In [24]:
recommend_smart("Vidyarthi Bhavan")

,name,cuisines,rate,votes
234,Prems Graama Bhojanam,south indian,4.3,322
229,South Kitchen,south indian,4.3,275
237,Puliyogare Point,south indian,4.2,444
72,Sri Guru Kottureshwara Davangere Benne Dosa,south indian,4.1,558
160,Gama Gama,south indian,3.9,264
111,Vijayalakshmi,south indian,3.9,47
40,Havyaka Mess,south indian,3.9,28
150,By 2 Coffee,south indian,3.9,25
152,Davangere Butter Dosa Hotel,south indian,3.8,25
93,Mane Thindi,south indian,3.7,130


In [25]:
# Save your model + data
import pickle

pickle.dump(df, open('restaurant_data.pkl', 'wb'))
pickle.dump(cosine_sim, open('similarity.pkl', 'wb'))

In [26]:
food_to_cuisine = {
    "pizza": "italian",
    "pasta": "italian",
    "biryani": "north indian",
    "dosa": "south indian",
    "idli": "south indian",
    "noodles": "chinese",
    "fried rice": "chinese",
    "burger": "fast food"
}

In [30]:
def recommend_food_swiggy_style(food, location=None, max_cost=None):

    cuisine = food_to_cuisine.get(food.lower(), food.lower())

    result = df[df['cuisines'].str.contains(cuisine)].copy()

    if location:
        result = result[result['location'].str.lower() == location.lower()]

    if max_cost:
        result = result[result['approx_cost(for two people)'] <= max_cost]

    result.loc[:, 'score'] = (result['rate'] * 2) + (result['votes'] / 1000)

    result = result.sort_values(by='score', ascending=False)

    return result[['name', 'location', 'cuisines', 'rate', 'votes', 'approx_cost(for two people)']].head(10)

In [32]:
recommend_food_swiggy_style("dosa", location="Banashankari", max_cost=300)

,name,location,cuisines,rate,votes,approx_cost(for two people)
425,Taaza Thindi,Banashankari,south indian,4.7,651,100.0
394,Ayodhya Upachar,Banashankari,"south indian, north indian, chinese, street food",4.3,734,200.0
426,Sri Laxmi Venkateshwara Coffee Bar,Banashankari,south indian,4.4,343,100.0
240,Udupi Ruchi Grand,Banashankari,"south indian, north indian, chinese, street food",4.2,60,200.0
103,Bengaluru Coffee House,Banashankari,"south indian, north indian, chinese, street food",4.1,201,300.0
436,SLV Refreshment,Banashankari,south indian,4.1,94,100.0
160,Gama Gama,Banashankari,south indian,3.9,264,200.0
450,Kavali,Banashankari,"south indian, street food",4.0,42,300.0
38,Maruthi Davangere Benne Dosa,Banashankari,south indian,4.0,17,150.0
429,Hotel Mangala,Banashankari,south indian,3.9,88,250.0
